# Feature Engineering — 20 Advanced Behavioral Features

This notebook computes **20 customer-level features** across **5 behavioral families** from the raw UCI Online Retail dataset. The pipeline:

1. Clean raw data (dedup, fix prices, parse dates)
2. Separate cancellation invoices (needed for return features)
3. Compute 20 features grouped by: Purchase Rhythm, Spend Shape, Basket Behavior, Volume & B2B, Customer Lifecycle
4. Separate B2B customers (SKU_HHI > 0.5)
5. Greedy Spearman correlation filter (threshold = 0.85) → ~14 retained features
6. QuantileTransformer → StandardScaler → clip to [-3, 3]
7. Family-level weighting (1/√n per family) for balanced Euclidean distance
8. Save to `data/processed/retail_features_clean.csv`

In [ ]:
# Cell 1: Imports & Load Raw Data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from IPython.display import display

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

DATA_PATH = '../data/raw/ecommerce-data.csv'
df = pd.read_csv(DATA_PATH, encoding='unicode_escape')
print(f"Raw data shape: {df.shape}")
df.head(3)

In [ ]:
# Cell 2: Data Cleaning (Inline)
# Keep cancellation invoices — they are needed for ReturnRate / ReturnValueRate

print(f"Initial shape: {df.shape}")

# 1. Drop full-row duplicates
df.drop_duplicates(keep='first', inplace=True)
print(f"After dedup: {df.shape}")

# 2. Parse InvoiceDate
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 3. Fill UnitPrice = 0 with mode per StockCode, then drop remaining zeros
df.loc[df['UnitPrice'] <= 0, 'UnitPrice'] = np.nan
mode_prices = df.groupby('StockCode')['UnitPrice'].transform(
    lambda x: x.mode()[0] if not x.mode().empty else np.nan
)
df['UnitPrice'] = df['UnitPrice'].fillna(mode_prices)
df.dropna(subset=['UnitPrice'], inplace=True)
print(f"After UnitPrice fix: {df.shape}")

# 4. Fill Description from StockCode mapping, drop remaining nulls
desc_mapping = (
    df.dropna(subset=['Description'])
    .drop_duplicates('StockCode')
    .set_index('StockCode')['Description']
)
df['Description'] = df['Description'].fillna(df['StockCode'].map(desc_mapping))
df.dropna(subset=['Description'], inplace=True)

# 5. Drop null CustomerID — cannot segment anonymous customers
df.dropna(subset=['CustomerID'], inplace=True)
df['CustomerID'] = df['CustomerID'].astype(int)
print(f"After dropping null CustomerID: {df.shape}")

# 6. Create TotalPrice
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print(f"\nFinal cleaned shape: {df.shape}")
print(f"Unique customers: {df['CustomerID'].nunique()}")
print(f"Date range: {df['InvoiceDate'].min()} → {df['InvoiceDate'].max()}")

In [ ]:
# Cell 3: Separate Cancellations for Return Features
# Cancellation invoices start with 'C'

is_cancel = df['InvoiceNo'].astype(str).str.startswith('C')

df_cancels = df[is_cancel].copy()
df_sales = df[~is_cancel].copy()

# For sales, enforce positive Quantity (should already be, but be safe)
df_sales = df_sales[df_sales['Quantity'] > 0]

print(f"Sales transactions: {df_sales.shape[0]:,}")
print(f"Cancellation transactions: {df_cancels.shape[0]:,}")
print(f"Customers in sales: {df_sales['CustomerID'].nunique():,}")
print(f"Customers with cancellations: {df_cancels['CustomerID'].nunique():,}")

In [ ]:
# Cell 4: Calculate 20 Advanced Features
# Reference date = 1 day after last transaction
reference_date = df_sales['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Reference date: {reference_date}")

# ------------------------------------------------------------------
# Helper: coefficient of variation (returns 0 if std or mean is 0)
# ------------------------------------------------------------------
def cv(series):
    """Coefficient of variation, safe for constant series."""
    m = series.mean()
    if m == 0 or len(series) < 2:
        return 0.0
    return series.std() / m

# ------------------------------------------------------------------
# Helper: Gini coefficient
# ------------------------------------------------------------------
def gini(values):
    """Gini coefficient of a list of values."""
    arr = np.sort(np.array(values, dtype=float))
    n = len(arr)
    if n == 0 or arr.sum() == 0:
        return 0.0
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * arr) - (n + 1) * np.sum(arr)) / (n * np.sum(arr))

# ==================================================================
# Build per-order summaries first (for order-level features)
# ==================================================================
order_summary = df_sales.groupby(['CustomerID', 'InvoiceNo']).agg(
    order_date=('InvoiceDate', 'min'),
    order_total=('TotalPrice', 'sum'),
    line_count=('StockCode', 'count'),
    total_qty=('Quantity', 'sum'),
    unique_skus=('StockCode', 'nunique'),
).reset_index()

print(f"Order summary: {order_summary.shape[0]:,} orders")

# ==================================================================
# FAMILY 1: Purchase Rhythm
# ==================================================================
def purchase_rhythm_features(group):
    """Compute Recency, InterPurchaseCV, ActiveSpanDays, MonthlyOrderRate."""
    dates = group['order_date'].sort_values()
    n_orders = len(dates)
    last_date = dates.iloc[-1]
    first_date = dates.iloc[0]

    recency = (reference_date - last_date).days
    active_span = (last_date - first_date).days

    if n_orders < 2:
        inter_cv = 0.0
        monthly_rate = float(n_orders)  # treat as 1 month
    else:
        gaps = dates.diff().dt.days.dropna()
        inter_cv = cv(gaps)
        months = max(active_span / 30.44, 1.0)
        monthly_rate = n_orders / months

    return pd.Series({
        'Recency': recency,
        'InterPurchaseCV': inter_cv,
        'ActiveSpanDays': active_span,
        'MonthlyOrderRate': monthly_rate,
    })

print("Computing Purchase Rhythm features...")
f_rhythm = order_summary.groupby('CustomerID').apply(
    purchase_rhythm_features, include_groups=False
)

# ==================================================================
# FAMILY 2: Spend Shape
# ==================================================================
def spend_shape_features(group):
    """Compute AOV, SpendGini, PriceCV, MaxSpendShare, SpendAcceleration."""
    totals = group['order_total'].values
    n_orders = len(totals)
    total_spend = totals.sum()

    aov = total_spend / n_orders if n_orders > 0 else 0.0
    spend_gini = gini(totals)
    max_spend_share = totals.max() / total_spend if total_spend > 0 else 1.0

    # SpendAcceleration: spend in 2nd half / spend in 1st half of active span
    if n_orders < 2:
        spend_accel = 1.0
    else:
        dates = group['order_date'].sort_values().values
        midpoint = dates[0] + (dates[-1] - dates[0]) / 2
        first_half = totals[group['order_date'].values <= midpoint].sum()
        second_half = totals[group['order_date'].values > midpoint].sum()
        spend_accel = second_half / max(first_half, 1e-6)

    return pd.Series({
        'AOV': aov,
        'SpendGini': spend_gini,
        'MaxSpendShare': max_spend_share,
        'SpendAcceleration': spend_accel,
    })

# PriceCV is computed from line-item level, not order level
print("Computing Spend Shape features...")
f_spend = order_summary.groupby('CustomerID').apply(
    spend_shape_features, include_groups=False
)

# PriceCV: CV of UnitPrice across all line items per customer
price_cv = df_sales.groupby('CustomerID')['UnitPrice'].apply(cv).rename('PriceCV')
f_spend = f_spend.join(price_cv)

# ==================================================================
# FAMILY 3: Basket Behavior
# ==================================================================
def basket_behavior_features(group):
    """Compute RepeatSKUFraction, BasketSizeCV, MedianBasketQty, NewSKURate."""
    n_orders = len(group)

    # BasketSizeCV: CV of line-item count per order
    basket_cv = cv(group['line_count']) if n_orders > 1 else 0.0

    # MedianBasketQty: median total quantity per order
    median_qty = group['total_qty'].median()

    return pd.Series({
        'BasketSizeCV': basket_cv,
        'MedianBasketQty': median_qty,
    })

print("Computing Basket Behavior features...")
f_basket = order_summary.groupby('CustomerID').apply(
    basket_behavior_features, include_groups=False
)

# RepeatSKUFraction: fraction of distinct SKUs purchased more than once (line-item level)
sku_counts = df_sales.groupby('CustomerID')['StockCode'].value_counts()
repeat_sku = sku_counts.groupby('CustomerID').apply(
    lambda x: (x > 1).mean()
).rename('RepeatSKUFraction')
f_basket = f_basket.join(repeat_sku)

# NewSKURate: fraction of orders containing at least one never-before-seen SKU
def new_sku_rate(cust_df):
    """For a customer's sales (sorted by date), what fraction of orders have a new SKU?"""
    cust_df = cust_df.sort_values('InvoiceDate')
    seen = set()
    orders = cust_df.groupby('InvoiceNo')
    new_count = 0
    total_orders = 0
    for _, order in orders:
        skus = set(order['StockCode'])
        if skus - seen:  # at least one new SKU
            new_count += 1
        seen.update(skus)
        total_orders += 1
    return new_count / total_orders if total_orders > 0 else 1.0

new_sku = df_sales.groupby('CustomerID').apply(
    new_sku_rate, include_groups=False
).rename('NewSKURate')
f_basket = f_basket.join(new_sku)

# ==================================================================
# FAMILY 4: Volume & B2B
# ==================================================================
print("Computing Volume & B2B features...")

# BulkLineRate: fraction of line items with Quantity >= 24
bulk_rate = df_sales.groupby('CustomerID')['Quantity'].apply(
    lambda x: (x >= 24).mean()
).rename('BulkLineRate')

# QuantityCV: CV of Quantity across all line items
qty_cv = df_sales.groupby('CustomerID')['Quantity'].apply(cv).rename('QuantityCV')

# SKU_HHI: Herfindahl-Hirschman Index of SKU purchase shares
def hhi(sku_series):
    """HHI from a series of StockCodes."""
    shares = sku_series.value_counts(normalize=True)
    return (shares ** 2).sum()

sku_hhi = df_sales.groupby('CustomerID')['StockCode'].apply(hhi).rename('SKU_HHI')

# BurstIndex: max monthly order count / mean monthly order count
def burst_index(group):
    """Ratio of peak month's orders to average month's orders."""
    monthly = group['order_date'].dt.to_period('M').value_counts()
    if len(monthly) < 2:
        return 1.0
    return monthly.max() / monthly.mean()

burst = order_summary.groupby('CustomerID').apply(
    burst_index, include_groups=False
).rename('BurstIndex')

f_volume = pd.DataFrame({
    'BulkLineRate': bulk_rate,
    'QuantityCV': qty_cv,
    'SKU_HHI': sku_hhi,
    'BurstIndex': burst,
})

# ==================================================================
# FAMILY 5: Customer Lifecycle
# ==================================================================
print("Computing Customer Lifecycle features...")

# ReturnRate: cancellation invoices / total invoices (sales + cancels)
sales_invoice_count = df_sales.groupby('CustomerID')['InvoiceNo'].nunique()
cancel_invoice_count = df_cancels.groupby('CustomerID')['InvoiceNo'].nunique()

# Align: fill NaN with 0 for customers with no cancellations
all_customers = sales_invoice_count.index
cancel_aligned = cancel_invoice_count.reindex(all_customers, fill_value=0)
total_invoices = sales_invoice_count + cancel_aligned
return_rate = (cancel_aligned / total_invoices).rename('ReturnRate')

# ReturnValueRate: |cancellation value| / total sales value
sales_value = df_sales.groupby('CustomerID')['TotalPrice'].sum()
cancel_value = df_cancels.groupby('CustomerID')['TotalPrice'].sum().abs()
cancel_value_aligned = cancel_value.reindex(all_customers, fill_value=0)
return_value_rate = (cancel_value_aligned / sales_value.clip(lower=1e-6)).rename('ReturnValueRate')

# QuarterConcentration: max quarterly order share
def quarter_concentration(group):
    """Max share of orders in any calendar quarter."""
    quarters = group['order_date'].dt.quarter.value_counts(normalize=True)
    return quarters.max()

q_conc = order_summary.groupby('CustomerID').apply(
    quarter_concentration, include_groups=False
).rename('QuarterConcentration')

f_lifecycle = pd.DataFrame({
    'ReturnRate': return_rate,
    'ReturnValueRate': return_value_rate,
    'QuarterConcentration': q_conc,
})

# ==================================================================
# Combine all 20 features
# ==================================================================
print("\nCombining all features...")
features = (
    f_rhythm
    .join(f_spend)
    .join(f_basket)
    .join(f_volume)
    .join(f_lifecycle)
)
features.index.name = 'CustomerID'

# Fill any remaining NaN (e.g., customers with no cancellations)
features = features.fillna(0)

print(f"Feature matrix shape: {features.shape}")
print(f"\nFeature columns ({len(features.columns)}):")
for i, col in enumerate(features.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Cell 5: Sanity Checks & Feature Distributions

# Descriptive stats
print("=" * 80)
print("DESCRIPTIVE STATISTICS")
print("=" * 80)
display(features.describe().T.round(4))

# Check for NaN / Inf
nan_count = features.isna().sum().sum()
inf_count = np.isinf(features.values).sum()
print(f"\nNaN values: {nan_count}")
print(f"Inf values: {inf_count}")
assert nan_count == 0, "NaN values found!"
assert inf_count == 0, "Inf values found!"
print("✅ No NaN or Inf values")

# Histogram grid (4×5) of raw feature distributions
fig, axes = plt.subplots(4, 5, figsize=(24, 16))
axes = axes.flatten()

for i, col in enumerate(features.columns):
    ax = axes[i]
    sns.histplot(features[col], kde=True, ax=ax, bins=50,
                 color='#3498db', edgecolor='white', alpha=0.7)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    # Add skewness annotation
    skew_val = features[col].skew()
    ax.annotate(f'skew={skew_val:.2f}', xy=(0.72, 0.92), xycoords='axes fraction',
                fontsize=8, color='red', fontweight='bold')

plt.suptitle('Raw Feature Distributions (20 Features)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: B2B Separation
# Filter: SKU_HHI > 0.5 → B2B cohort

B2B_THRESHOLD = 0.5

b2b_mask = features['SKU_HHI'] > B2B_THRESHOLD
df_b2b = features[b2b_mask].copy()
df_retail = features[~b2b_mask].copy()

print(f"B2B customers (SKU_HHI > {B2B_THRESHOLD}): {len(df_b2b):,}")
print(f"Retail customers: {len(df_retail):,}")
print(f"B2B share: {len(df_b2b) / len(features) * 100:.1f}%")

# Save B2B cohort
b2b_path = '../data/processed/b2b_customers.csv'
df_b2b.to_csv(b2b_path)
print(f"\n✅ B2B customers saved to {b2b_path}")

# Continue with retail cohort only
print(f"\nRetail cohort shape for clustering: {df_retail.shape}")

In [ ]:
# Cell 7: Greedy Correlation Filter
# Threshold: |Spearman correlation| > 0.85 → drop the non-protected feature
# with the highest mean absolute correlation

CORR_THRESHOLD = 0.85
PROTECTED_FEATURES = ['Recency', 'MonthlyOrderRate', 'ReturnRate',
                       'QuarterConcentration', 'SKU_HHI']

# Compute Spearman correlation
corr_matrix = df_retail.corr(method='spearman').abs()

# Visualize before filtering
fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0.5, ax=ax, square=True, linewidths=0.5,
            cbar_kws={'label': '|Spearman ρ|'})
ax.set_title('Absolute Spearman Correlation Matrix (Before Filtering)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Greedy filter
features_to_keep = list(df_retail.columns)
dropped_features = []

while True:
    # Recompute correlation for remaining features
    corr_sub = df_retail[features_to_keep].corr(method='spearman').abs()
    # Zero the diagonal using pandas (pandas 3.x .values is read-only)
    for col in corr_sub.columns:
        corr_sub.loc[col, col] = 0.0

    # Find max correlation pair
    max_corr = corr_sub.max().max()
    if max_corr <= CORR_THRESHOLD:
        break

    # Find the pair
    pair = corr_sub.stack().idxmax()
    feat_a, feat_b = pair

    # Decide which to drop: prefer non-protected, then highest mean |corr|
    a_protected = feat_a in PROTECTED_FEATURES
    b_protected = feat_b in PROTECTED_FEATURES

    if a_protected and b_protected:
        # Both protected — drop the one with higher mean correlation to others
        mean_a = corr_sub[feat_a].mean()
        mean_b = corr_sub[feat_b].mean()
        to_drop = feat_a if mean_a > mean_b else feat_b
    elif a_protected:
        to_drop = feat_b
    elif b_protected:
        to_drop = feat_a
    else:
        # Neither protected — drop the one with higher mean correlation
        mean_a = corr_sub[feat_a].mean()
        mean_b = corr_sub[feat_b].mean()
        to_drop = feat_a if mean_a > mean_b else feat_b

    features_to_keep.remove(to_drop)
    dropped_features.append((to_drop, feat_a, feat_b, max_corr))
    print(f"  Dropped '{to_drop}' (|corr({feat_a}, {feat_b})| = {max_corr:.3f})")

print(f"\n{'=' * 60}")
print(f"CORRELATION FILTER RESULTS")
print(f"{'=' * 60}")
print(f"Features before: {len(df_retail.columns)}")
print(f"Features after:  {len(features_to_keep)}")
print(f"Dropped ({len(dropped_features)}): {[d[0] for d in dropped_features]}")
print(f"\nRetained features:")
for i, f in enumerate(features_to_keep, 1):
    protected = ' (protected)' if f in PROTECTED_FEATURES else ''
    print(f"  {i:2d}. {f}{protected}")

# Apply filter
df_filtered = df_retail[features_to_keep].copy()

In [ ]:
# Cell 8: Advanced Transformation Pipeline
# Step 1: QuantileTransformer(output_distribution='normal')
# Step 2: StandardScaler() — ensure exact mean=0, std=1
# Step 3: np.clip(values, -3.0, 3.0) — hard clip extremes

feature_cols = df_filtered.columns.tolist()

# Step 1: QuantileTransformer
qt = QuantileTransformer(output_distribution='normal', random_state=42)
df_qt = pd.DataFrame(
    qt.fit_transform(df_filtered),
    columns=feature_cols,
    index=df_filtered.index
)

# Step 2: StandardScaler
scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_qt),
    columns=feature_cols,
    index=df_filtered.index
)

# Step 3: Clip to [-3, 3]
df_transformed = df_scaled.clip(lower=-3.0, upper=3.0)

# Verify transformation results
print("=" * 70)
print("TRANSFORMATION VERIFICATION")
print("=" * 70)
stats = pd.DataFrame({
    'mean': df_transformed.mean(),
    'std': df_transformed.std(),
    'min': df_transformed.min(),
    'max': df_transformed.max(),
}).round(4)
display(stats)

# Verify all values in [-3, 3]
assert df_transformed.min().min() >= -3.0, "Values below -3.0!"
assert df_transformed.max().max() <= 3.0, "Values above 3.0!"
print("\n✅ All values in [-3.0, 3.0]")

# Histogram grid of transformed features
n_features = len(feature_cols)
n_cols_plot = 5
n_rows_plot = (n_features + n_cols_plot - 1) // n_cols_plot

fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(24, 4 * n_rows_plot))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    sns.histplot(df_transformed[col], kde=True, ax=ax, bins=50,
                 color='#2ecc71', edgecolor='white', alpha=0.7)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_xlabel('')

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Transformed Feature Distributions (QuantileTransformer → StandardScaler → Clip)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: Family Weighting
# Weight = 1 / sqrt(n_features_in_family)
# Ensures each of the 5 behavioral dimensions contributes equally in Euclidean distance

FEATURE_FAMILIES = {
    'Purchase Rhythm': ['Recency', 'InterPurchaseCV', 'ActiveSpanDays', 'MonthlyOrderRate'],
    'Spend Shape': ['AOV', 'SpendGini', 'PriceCV', 'MaxSpendShare', 'SpendAcceleration'],
    'Basket Behavior': ['RepeatSKUFraction', 'BasketSizeCV', 'MedianBasketQty', 'NewSKURate'],
    'Volume & B2B': ['BulkLineRate', 'QuantityCV', 'SKU_HHI', 'BurstIndex'],
    'Customer Lifecycle': ['ReturnRate', 'ReturnValueRate', 'QuarterConcentration'],
}

# Build weight lookup for retained features only
retained_features = df_transformed.columns.tolist()
weight_dict = {}

print(f"{'Family':<25s} {'Retained Features':<55s} {'n':>3s}  {'Weight':>8s}")
print("=" * 95)

for family, members in FEATURE_FAMILIES.items():
    # Only count retained members
    retained_members = [f for f in members if f in retained_features]
    n = len(retained_members)
    if n > 0:
        w = 1.0 / np.sqrt(n)
        for f in retained_members:
            weight_dict[f] = w
        print(f"{family:<25s} {str(retained_members):<55s} {n:3d}  {w:8.4f}")

print(f"\nTotal weighted features: {len(weight_dict)}")

# Apply weights
df_weighted = df_transformed.copy()
for col in df_weighted.columns:
    df_weighted[col] *= weight_dict.get(col, 1.0)

# Verify: show weighted feature variances (should be ~equal within families)
print(f"\n{'Feature':<25s} {'Weight':>8s} {'Weighted Var':>14s}")
print("-" * 50)
for col in df_weighted.columns:
    w = weight_dict.get(col, 1.0)
    var = df_weighted[col].var()
    print(f"{col:<25s} {w:8.4f} {var:14.4f}")

In [ ]:
# Cell 10: Save Output
# Save the retail cohort (scaled + weighted, ~14 features + CustomerID)

output_path = '../data/processed/retail_features_clean.csv'
df_weighted.to_csv(output_path)

# Final sanity checks
print("=" * 70)
print("FINAL OUTPUT SUMMARY")
print("=" * 70)
print(f"Output path: {output_path}")
print(f"Shape: {df_weighted.shape}")
print(f"NaN count: {df_weighted.isna().sum().sum()}")
print(f"Inf count: {np.isinf(df_weighted.values).sum()}")
print(f"\nDescriptive statistics:")
display(df_weighted.describe().T.round(4))

print(f"\n✅ Retail features saved to {output_path}")
print(f"\nReady for Phase 3: K-Means clustering")